# 04 — Custom Data & Training (Google Colab, T4 GPU)

**Deliverable 4** of the PPE Compliance Monitor capstone.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ksamaani/ppe-compliance-monitor/blob/main/notebooks/04_training_colab.ipynb)

---

## How to run this

1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. **Runtime → Run all.**
3. Wait ~40–60 minutes. Two full training runs happen back to back.
4. The final cell zips the artefacts and starts a browser download. Keep that zip.

The development machine for this project has no CUDA device, so this is the one stage that has
to run elsewhere. Every API call below was first exercised on CPU by
`scripts/smoke_test_training.py` against a 24-image subset — see `logs/05_smoke_test.log` —
so this notebook should not fail on a typo 30 minutes in.

---

## What is being trained, and why two runs

Notebook 01 established the gap: COCO-pretrained weights find *people* but have no concept of a
hard hat, so compliance cannot be assessed at all. This notebook closes that gap by fine-tuning
`yolo11n` on 2,605 labelled construction-site images covering 10 PPE classes.

Two runs, because a single run tells you nothing about *why* it landed where it did:

| knob | Run A — baseline | Run B — tuned | reasoning |
|---|---|---|---|
| `epochs` | 25 | 30 (`patience=10`) | give B headroom, stop early if it stalls |
| `freeze` | 0 | **10** | COCO's early layers already encode edges and texture; freezing them keeps generic features and spends the gradient budget on the PPE-specific head |
| `weight_decay` | 0.0005 | **0.001** | stronger L2 — 2.6k images is small enough to memorise |
| `hsv_v` | 0.4 | **0.5** | site lighting swings between glare and deep shadow |
| `degrees` | 0.0 | **10.0** | fixed site cameras are never perfectly level |
| `translate` | 0.1 | **0.2** | workers appear anywhere in frame, not centred |
| `scale` | 0.5 | **0.7** | huge distance range between near and far workers |
| `close_mosaic` | 10 | 10 | mosaic off for the last 10 epochs so the model finishes on realistic whole images |

Both runs use `seed=42` and `imgsz=640` so the comparison isolates the knobs above.

In [ ]:
# --- 1. Confirm we actually have a GPU ---------------------------------------
!nvidia-smi

import torch
print()
print("torch      :", torch.__version__)
print("cuda avail :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device     :", torch.cuda.get_device_name(0))
else:
    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.")

In [ ]:
# --- 2. Dependencies ---------------------------------------------------------
%pip install -q ultralytics kagglehub

import ultralytics
ultralytics.checks()

In [ ]:
# --- 3. Pull the project code -------------------------------------------------
# The dataset helpers live in the repo so that this notebook, the local smoke
# test and the evaluation notebook all resolve splits and class order the SAME
# way. A mismatched class order trains happily and mislabels every metric.
import sys
from pathlib import Path

REPO = Path("/content/ppe-compliance-monitor")
if not REPO.exists():
    !git clone -q https://github.com/Ksamaani/ppe-compliance-monitor.git {REPO}
sys.path.insert(0, str(REPO))

from ppe_monitor import dataset
from ppe_monitor.dataset import CLASS_NAMES

print("classes:", CLASS_NAMES)

In [ ]:
# --- 4. Dataset --------------------------------------------------------------
root = dataset.download()
split_root = dataset.find_split_root(root)
splits = dataset.resolve_splits(split_root)

report = dataset.describe(splits)
for name, stats in report.items():
    print(f"{name}: {stats['images']} images, {stats['instances']} instances")
    for cls, n in sorted(stats["per_class"].items(), key=lambda kv: -kv[1]):
        print(f"    {cls:<16}{n:>7}")

DATA_YAML = dataset.write_data_yaml(split_root, Path("/content/data.yaml"), splits)
print()
print(DATA_YAML.read_text())

### A note on what the class counts mean for this project

`NO-Hardhat` is the class this system exists to catch, and it has ~2,300 training instances —
enough to learn properly. The rarer classes (`Mask`, `vehicle`) have far fewer and will score
lower; that is a property of the dataset, not of the training run, and the evaluation notebook
reports them separately rather than hiding them inside a single mAP number.

In [ ]:
# --- 5. Run A: baseline -------------------------------------------------------
import time
from ultralytics import YOLO

t0 = time.time()
model_a = YOLO("yolo11n.pt")
results_a = model_a.train(
    data=str(DATA_YAML),
    epochs=25,
    imgsz=640,
    batch=-1,          # AutoBatch: fill the T4's memory
    seed=42,
    project="/content/runs",
    name="run_a_baseline",
    exist_ok=True,
    plots=True,
)
print(f"\nRun A finished in {(time.time() - t0) / 60:.1f} min -> {results_a.save_dir}")

In [ ]:
# --- 6. Run A: validation ------------------------------------------------------
best_a = Path(results_a.save_dir) / "weights" / "best.pt"
metrics_a = YOLO(str(best_a)).val(
    data=str(DATA_YAML), imgsz=640, split="val",
    project="/content/runs", name="run_a_val", exist_ok=True, plots=True,
)

print(f"mAP50    : {metrics_a.box.map50:.4f}")
print(f"mAP50-95 : {metrics_a.box.map:.4f}")
print(f"precision: {metrics_a.box.mp:.4f}")
print(f"recall   : {metrics_a.box.mr:.4f}")

In [ ]:
# --- 7. Run B: tuned ----------------------------------------------------------
t0 = time.time()
model_b = YOLO("yolo11n.pt")
results_b = model_b.train(
    data=str(DATA_YAML),
    epochs=30,
    imgsz=640,
    batch=-1,
    seed=42,
    patience=10,        # stop if val mAP stalls for 10 epochs
    freeze=10,          # keep COCO's generic backbone features
    weight_decay=0.001, # stronger L2 against memorising 2.6k images
    hsv_v=0.5,          # glare and deep shadow on site
    degrees=10.0,       # cameras are never perfectly level
    translate=0.2,      # workers are not centred
    scale=0.7,          # wide range of worker distances
    close_mosaic=10,    # finish on realistic whole images
    project="/content/runs",
    name="run_b_tuned",
    exist_ok=True,
    plots=True,
)
print(f"\nRun B finished in {(time.time() - t0) / 60:.1f} min -> {results_b.save_dir}")

In [ ]:
# --- 8. Run B: validation ------------------------------------------------------
best_b = Path(results_b.save_dir) / "weights" / "best.pt"
metrics_b = YOLO(str(best_b)).val(
    data=str(DATA_YAML), imgsz=640, split="val",
    project="/content/runs", name="run_b_val", exist_ok=True, plots=True,
)

print(f"mAP50    : {metrics_b.box.map50:.4f}")
print(f"mAP50-95 : {metrics_b.box.map:.4f}")
print(f"precision: {metrics_b.box.mp:.4f}")
print(f"recall   : {metrics_b.box.mr:.4f}")

## 9. Did the tuning help, and did either run overfit?

The comparison below is the actual deliverable — not the raw numbers, but the reading of them.
Look for three things:

- **`train/box_loss` still falling while `val/box_loss` turns upward** → overfitting; the model
  is memorising the training images.
- **Both losses still falling at the last epoch** → underfitting; it wanted more epochs.
- **`NO-Hardhat` recall specifically**, not just overall mAP. This system is judged on catching
  violations, and overall mAP is dominated by the common `Person` class.

In [ ]:
# --- 9. Curves ----------------------------------------------------------------
import pandas as pd
import matplotlib.pyplot as plt

def load_curve(save_dir):
    df = pd.read_csv(Path(save_dir) / "results.csv")
    df.columns = [c.strip() for c in df.columns]
    return df

df_a, df_b = load_curve(results_a.save_dir), load_curve(results_b.save_dir)
print(f"Run A epochs completed: {len(df_a)}")
print(f"Run B epochs completed: {len(df_b)}  (patience=10 may have stopped it early)")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

for df, label in ((df_a, "A baseline"), (df_b, "B tuned")):
    axes[0].plot(df["epoch"], df["train/box_loss"], label=f"{label} train")
    axes[0].plot(df["epoch"], df["val/box_loss"], "--", label=f"{label} val")
    axes[1].plot(df["epoch"], df["metrics/mAP50(B)"], label=label)
    axes[2].plot(df["epoch"], df["metrics/mAP50-95(B)"], label=label)

axes[0].set_title("box loss - train (solid) vs val (dashed)")
axes[1].set_title("mAP50")
axes[2].set_title("mAP50-95")
for ax in axes:
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig("/content/training_curves.png", dpi=110)
plt.show()

In [ ]:
# --- 10. Per-class comparison --------------------------------------------------
rows = []
for label, m in (("A baseline", metrics_a), ("B tuned", metrics_b)):
    for i, cls_idx in enumerate(m.box.ap_class_index):
        p, r, ap50, ap = m.box.class_result(i)
        rows.append({
            "run": label,
            "class": CLASS_NAMES[int(cls_idx)],
            "precision": round(float(p), 4),
            "recall": round(float(r), 4),
            "mAP50": round(float(ap50), 4),
            "mAP50-95": round(float(ap), 4),
        })

per_class = pd.DataFrame(rows)
pivot = per_class.pivot(index="class", columns="run", values=["recall", "mAP50"])
print("Recall and mAP50 by class:\n")
print(pivot.to_string())

print("\n--- the classes this project is actually judged on ---")
critical = per_class[per_class["class"].str.startswith("NO-")]
print(critical.to_string(index=False))

In [ ]:
# --- 11. Summary table ---------------------------------------------------------
summary = pd.DataFrame([
    {
        "run": "A baseline",
        "epochs_run": len(df_a),
        "mAP50": round(float(metrics_a.box.map50), 4),
        "mAP50-95": round(float(metrics_a.box.map), 4),
        "precision": round(float(metrics_a.box.mp), 4),
        "recall": round(float(metrics_a.box.mr), 4),
    },
    {
        "run": "B tuned",
        "epochs_run": len(df_b),
        "mAP50": round(float(metrics_b.box.map50), 4),
        "mAP50-95": round(float(metrics_b.box.map), 4),
        "precision": round(float(metrics_b.box.mp), 4),
        "recall": round(float(metrics_b.box.mr), 4),
    },
])
print(summary.to_string(index=False))

winner = "B tuned" if metrics_b.box.map > metrics_a.box.map else "A baseline"
winner_dir = results_b.save_dir if winner == "B tuned" else results_a.save_dir
print(f"\nSelected on mAP50-95: {winner}")
print("(mAP50-95 rather than mAP50: localisation quality matters when the box is")
print(" cropped as photographic evidence of a violation.)")

In [ ]:
# --- 12. Package everything for download ---------------------------------------
import json
import shutil

OUT = Path("/content/ppe_training_artifacts")
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True)

for label, save_dir, m, df in (
    ("run_a_baseline", results_a.save_dir, metrics_a, df_a),
    ("run_b_tuned", results_b.save_dir, metrics_b, df_b),
):
    dst = OUT / label
    dst.mkdir(parents=True)
    src = Path(save_dir)
    shutil.copy2(src / "weights" / "best.pt", dst / "best.pt")
    for artefact in ("results.csv", "args.yaml", "confusion_matrix.png",
                     "confusion_matrix_normalized.png", "results.png",
                     "PR_curve.png", "P_curve.png", "R_curve.png", "F1_curve.png",
                     "labels.jpg"):
        if (src / artefact).exists():
            shutil.copy2(src / artefact, dst / artefact)
    json.dump({
        "run": label,
        "epochs_run": int(len(df)),
        "mAP50": float(m.box.map50),
        "mAP50_95": float(m.box.map),
        "precision": float(m.box.mp),
        "recall": float(m.box.mr),
        "per_class": {
            CLASS_NAMES[int(c)]: dict(zip(
                ("precision", "recall", "mAP50", "mAP50_95"),
                [float(v) for v in m.box.class_result(i)],
            ))
            for i, c in enumerate(m.box.ap_class_index)
        },
    }, open(dst / "metrics.json", "w"), indent=2)

# The winning weights, at the path the rest of the project expects.
shutil.copy2(Path(winner_dir) / "weights" / "best.pt", OUT / "best.pt")
shutil.copy2("/content/training_curves.png", OUT / "training_curves.png")
per_class.to_csv(OUT / "per_class_comparison.csv", index=False)
summary.to_csv(OUT / "run_summary.csv", index=False)
json.dump({"winner": winner, "selected_on": "mAP50-95"}, open(OUT / "selection.json", "w"), indent=2)

archive = shutil.make_archive("/content/ppe_training_artifacts", "zip", OUT)
print(f"{archive}  ({Path(archive).stat().st_size / 1e6:.1f} MB)")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(OUT)}")

In [ ]:
# --- 13. Download --------------------------------------------------------------
from google.colab import files
files.download("/content/ppe_training_artifacts.zip")

## What to do with the zip

Unpack it into the repository:

```
ppe_training_artifacts/best.pt          ->  models/best.pt
ppe_training_artifacts/                 ->  runs_artifacts/
```

Then, back on the local machine:

- `03_evaluation.ipynb` runs `model.val` on the **held-out test split** and sweeps the confidence
  threshold to pick the operating point this system ships with.
- `02_video_analytics.ipynb` is re-run so the violation ledger works on real PPE classes instead
  of COCO's `person`.
- `scripts/export_onnx.py` exports the deployment model.

If the browser download was blocked, the zip is also at `/content/ppe_training_artifacts.zip`
in the Colab file browser (folder icon in the left sidebar).